# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the [FAIR²](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. We'll walk through metadata inspection, overview of record sets and fields, loading the dataset, and basic exploratory analysis.

### Dataset Source
- **Croissant schema URL:** [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and inspect the dataset description using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pd
# Define the Croissant schema URLcroissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load the dataset using mlcroissantdataset = mlc.Dataset(croissant_url)
# Print high-level metadataprint(f"Dataset: {dataset.metadata.name}\n")print(f"Description: {dataset.metadata.description}\n")print(f"Version: {dataset.metadata.version}")print(f"Published: {dataset.metadata.datePublished}")print(f"License: {dataset.metadata.license}")

## 2. Data Overview

Review available **record sets**, their `@id`s, and the fields within each record set. We'll print an overview of the data structure as defined in Croissant schema using the API.

In [ ]:
# List available record sets (`cr:RecordSet`) in the datasetrecord_sets = list(dataset.metadata.recordSets)print(f"Found {len(record_sets)} record sets.\n")
for idx, rs in enumerate(record_sets):    print(f"{idx+1}. RecordSet name: {rs.name}")    print(f"   @id: {rs.id}")    print(f"   Description: {rs.description if rs.description else 'No description'}")    print(f"   Fields (and their @id's):")    for f in rs.fields:        print(f"      - {f.name} (@id: {f.id}, type: {f.dataType if hasattr(f, 'dataType') else 'n/a'})")    print("")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. **Use only `@id` values** for all entities.

You can repeat this for any number of record sets by updating `record_set_ids`.

In [ ]:
# Prepare list of record set @id's to extract (modify if more desired)record_set_ids = [rs.id for rs in record_sets]
# For demonstration, target the main protein data table, which is usually the first/only real tabledataframes = {}
for record_set_id in record_set_ids:    print(f"Loading records from RecordSet @id: {record_set_id}")    records = list(dataset.records(record_set=record_set_id))    if records:        df = pd.DataFrame(records)        dataframes[record_set_id] = df        print(f"Loaded {len(df)} records, columns: {df.columns.tolist()}\n")    else:        print("No records loaded for this record set.\n")
# Let's display the first few rows of the main record set (assuming the first one is the main table)main_record_set_id = record_set_ids[0]print(f"Columns:\n{dataframes[main_record_set_id].columns.tolist()}")dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps: filtering, normalizing, and grouping. 

> **Note**: All field (column) references are by their Croissant `@id` if possible. The printed table above provides exact names (often `@id`) for each column.

We'll pick a numeric field for basic analysis, for instance, coverage, peptide count, or MW (molecular weight) depending on availability.

In [ ]:
# Choose a numeric field by @id, e.g., 'coverage', 'MW', 'unique_peptides', etc.
main_df = dataframes[main_record_set_id]

# Example: try to auto-select a likely numeric field
numeric_field_candidates = [col for col in main_df.columns if any(x in col.lower() for x in ['coverage', 'mw', 'peptide', 'count', 'abund', 'number'])]
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
else:
    # Fallback: just pick first numeric column
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field = col
            break
    else:
        raise RuntimeError('No obvious numeric field found!')

print(f"Selected numeric field (@id): {numeric_field}")

# Filtering: e.g., show records where numeric_field > threshold
threshold = main_df[numeric_field].quantile(0.75) # Use 75th percentile as a reasonable threshold
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (Top 25%):")
print(filtered_df.head())

# Normalization
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping: try to group by a likely categorical field (e.g., 'sample', 'accession', 'category')
group_field_candidates = [col for col in main_df.columns if any(x in col.lower() for x in ['sample', 'modification', 'group', 'category', 'accession'])]
if group_field_candidates:
    group_field = group_field_candidates[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"\nGrouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())
else:
    print("\nNo obvious group-by field found.")

## 5. Visualization

Visualize the numeric field's distribution and its relation to a group, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If grouping field is available, plot boxplot
if 'group_field' in locals():
    # Limit number of categories for readability
    top_groups = filtered_df[group_field].value_counts().head(10).index
    subset = filtered_df[filtered_df[group_field].isin(top_groups)]
    plt.figure(figsize=(10,6))
    sns.boxplot(data=subset, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- We loaded the [FAIR² dataset](https://doi.org/10.71728/senscience.vq4a-28xa) Croissant schema and explored its main data record set.
- The dataset offers mass spectrometry protein abundance and characteristic fields for extracellular vesicles from human mast cells.
- Using only `@id` fields, we demonstrated core filtering and normalization operations as well as straightforward visualizations.

> For advanced or domain-specific analyses, consult the schema's documentation at the provided Croissant URL and review available field and record set definitions via the `mlcroissant` metadata API.